# Advanced 14 — Concentration Inequalities and Large Deviations

Practice notebook for the **"Concentration Inequalities and Large Deviations"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Hoeffding's Inequality

**Theorem (Hoeffding).** Let \(X_1,\dots,X_n\) be independent with \(a_i \leq X_i \leq b_i\) almost surely and \(E[X_i]=\mu_i\). For \(S_n=\sum_{i=1}^n X_i\) and any \(t>0\),

\[
P\big(S_n - E[S_n] \geq t\big) \;\leq\; \exp\!\left(-\frac{2t^2}{\sum_{i=1}^n (b_i-a_i)^2}\right).
\]

For the i.i.d. bounded case \(X_i\in[a,b]\) with mean \(\mu\), applying the bound to the two-sided event via \(|\bar X_n-\mu|\) gives the familiar **sample-mean** form

\[
P\big(|\bar X_n - \mu| \geq t\big) \;\leq\; 2\exp\!\left(-\frac{2 n t^2}{(b-a)^2}\right).
\]

The tails shrink **exponentially in \(n\)** — much faster than the CLT's \(1/\sqrt{n}\) scaling suggests for fixed \(t\). Below we estimate the left-hand side by Monte Carlo for bounded uniform variables and overlay the Hoeffding upper bound as a function of \(t\).


In [ ]:
# Hoeffding: bounded variables on [a, b], sample-mean concentration
a, b = -1.0, 3.0          # support
mu = 0.5 * (a + b)         # true mean of Uniform(a,b)
n = 200                    # sample size
n_trials = 200_000
ts = np.linspace(0.02, 1.2, 40)

rng = np.random.default_rng(7)
samples = rng.uniform(a, b, size=(n_trials, n))          # (trials, n)
xbars = samples.mean(axis=1)                              # (trials,)

# Empirical two-sided tail: P(|Xbar_n - mu| >= t) for each t
emp = np.array([np.mean(np.abs(xbars - mu) >= t) for t in ts])

# Hoeffding two-sided bound: 2 exp(-2 n t^2 / (b-a)^2)
hoeff = 2.0 * np.exp(-2.0 * n * ts**2 / (b - a)**2)
hoeff = np.minimum(hoeff, 1.0)

plt.figure()
plt.semilogy(ts, np.maximum(emp, 1e-7), 'o-', label='Empirical', ms=4)
plt.semilogy(ts, hoeff, 'k--', lw=2, label=r'Hoeffding $2\exp(-2nt^2/(b-a)^2)$')
plt.xlabel('t'); plt.ylabel(r'$P(|\bar X_n - \mu| \geq t)$')
plt.title(f'Hoeffding bound vs simulation  (n={n}, Uniform[{a},{b}])')
plt.legend(); plt.tight_layout(); plt.show()

# Sanity print at one threshold
t0 = 0.5
print(f"t={t0}: empirical={np.mean(np.abs(xbars-mu)>=t0):.5f}, "
      f"Hoeffding={2*np.exp(-2*n*t0**2/(b-a)**2):.5f}")


---
## Part 2 — Chernoff Bounds for Sub-Gaussian Sums

The Chernoff method uses the **moment generating function** \(M(\theta)=E[e^{\theta X}]\). For any \(\theta>0\),

\[
P(S_n \geq n a) \;\leq\; \exp\!\big(-n\,\sup_{\theta>0}\{\theta a - \log M(\theta)\}\big).
\]

If the \(X_i\) are \(\sigma^2\)-**sub-Gaussian** (so \(\log M(\theta)\leq \sigma^2\theta^2/2\)), optimizing over \(\theta\) yields the clean bound

\[
P\!\left(\frac{S_n}{n} \geq \mu + \varepsilon\right) \;\leq\; \exp\!\left(-\frac{n\varepsilon^2}{2\sigma^2}\right).
\]

Here we take Bernoulli\(p\) variables (bounded, hence sub-Gaussian with \(\sigma^2=\tfrac14\)) and verify the Chernoff/sub-Gaussian bound against simulation for several \(\varepsilon\).


In [ ]:
# Chernoff / sub-Gaussian bound for Bernoulli(p) sample mean
p = 0.3
mu = p
sigma2_sg = 0.25                 # Bernoulli is 1/4-sub-Gaussian
n = 400
n_trials = 300_000

eps_grid = np.linspace(0.02, 0.35, 25)

rng = np.random.default_rng(11)
S = rng.binomial(n, p, size=n_trials)            # sum of n Bernoulli(p)
xbars = S / n

emp = np.array([np.mean(xbars >= mu + e) for e in eps_grid])
chernoff = np.exp(-n * eps_grid**2 / (2 * sigma2_sg))

plt.figure()
plt.semilogy(eps_grid, np.maximum(emp, 1e-7), 'o-', ms=4, label='Empirical')
plt.semilogy(eps_grid, chernoff, 'k--', lw=2,
             label=r'Chernoff $\exp(-n\varepsilon^2/(2\sigma^2))$')
plt.xlabel(r'$\varepsilon$'); plt.ylabel(r'$P(\bar X_n \geq \mu + \varepsilon)$')
plt.title(f'Chernoff (sub-Gaussian) vs simulation  (Bernoulli p={p}, n={n})')
plt.legend(); plt.tight_layout(); plt.show()

# Numerical KL-divergence Chernoff bound (the exact Bernoulli form)
from scipy.special import rel_entr
def bern_kl(q, p):
    q = np.clip(q, 1e-12, 1-1e-12)
    p = np.clip(p, 1e-12, 1-1e-12)
    return q*np.log(q/p) + (1-q)*np.log((1-q)/(1-p))
kl_bound = np.exp(-n * bern_kl(mu + eps_grid, mu))
print(f"At eps=0.2: empirical={np.mean(xbars>=mu+0.2):.5e}, "
      f"subGauss={np.exp(-n*0.2**2/(2*sigma2_sg)):.5e}, "
      f"KL-Chernoff={np.exp(-n*bern_kl(mu+0.2,mu)):.5e}")


---
## Part 3 — Cramér's Theorem and the Rate Function

Cramér's theorem gives a **large deviations principle** for the sample mean of i.i.d. light-tailed variables. For \(S_n=\sum_{i=1}^n X_i\),

\[
P\!\left(\frac{S_n}{n} \approx x\right) \;\asymp\; \exp\!\big(-n\,I(x)\big),
\qquad
I(x) = \sup_{\theta\in\mathbb{R}} \{\theta x - \Lambda(\theta)\},
\quad \Lambda(\theta)=\log E[e^{\theta X_1}].
\]

The rate function \(I(x)\) is the **Legendre–Fenchel transform** of the cumulant generating function \(\Lambda\). It is far sharper than the CLT, which only describes \(O(1/\sqrt{n})\) fluctuations around the mean.

For bounded \(X_i\in[a,b]\) we can evaluate \(\Lambda(\theta)\) numerically by quadrature (or exactly for uniform). Below we:

1. compute \(\Lambda(\theta)\) and its Legendre transform \(I(x)\) numerically,
2. estimate \((1/n)\log P(\bar X_n \geq a)\) by **rare-event replication** at several \(n\),
3. compare the empirical rate against \(I(x)\) — they should align as \(n\) grows.


In [ ]:
# Cramer's theorem: rate function for bounded Uniform(a,b) variables
a, b = -1.0, 3.0
mu = 0.5 * (a + b)

# 1) Numerical log-MGF Lambda(theta) = log E[exp(theta X)] for Uniform(a,b):
#    E[exp(theta X)] = (exp(theta b) - exp(theta a)) / (theta (b-a))
def Lambda(theta):
    theta = np.asarray(theta, dtype=float)
    out = np.empty_like(theta)
    small = np.abs(theta) < 1e-8
    out[small] = mu * theta[small] + (b - a)**2 * theta[small]**2 / 24.0
    t = theta[~small]
    M = (np.exp(t * b) - np.exp(t * a)) / (t * (b - a))
    out[~small] = np.log(M)
    return out

thetas = np.linspace(-3.0, 3.0, 1001)
Lam = Lambda(thetas)

# 2) Rate function I(x) = sup_theta { theta*x - Lambda(theta) }
xs = np.linspace(a + 0.05, b - 0.05, 121)
I_of_x = np.array([np.max(thetas * x - Lam) for x in xs])
I_of_x = np.maximum(I_of_x, 0.0)

plt.figure()
plt.plot(xs, I_of_x, 'k-', lw=2, label=r'$I(x)=\sup_\theta\{\theta x-\Lambda(\theta)\}$')
plt.axvline(mu, color='r', ls=':', label=fr'$\mu={mu}$')
plt.xlabel('x'); plt.ylabel(r'$I(x)$')
plt.title('Cramér rate function for Uniform[-1, 3]')
plt.legend(); plt.tight_layout(); plt.show()

# 3) Rare-event estimation of (1/n) log P(Xbar_n >= x*) for several n
x_star = 1.5             # upper-tail threshold (> mu)
print(f"Threshold x*={x_star}, theoretical rate I(x*)="
      f"{np.max(thetas*x_star - Lam):.5f}")

ns = [20, 50, 100, 200, 400, 800]
rate_est = []
rng = np.random.default_rng(123)
for n in ns:
    # Crude MC with a large number of trials; use replication to reduce noise.
    trials = 4_000_000 if n <= 100 else 800_000
    # Draw in blocks to keep memory bounded
    hits = 0
    block = 200_000
    for _ in range(trials // block):
        S = rng.uniform(a, b, size=(block, n)).sum(axis=1)
        hits += int(np.sum(S / n >= x_star))
    p_hat = max(hits / trials, 1.0 / trials)
    rate = -np.log(p_hat) / n
    rate_est.append(rate)
    print(f"  n={n:4d}: P_hat={p_hat:.3e},  empirical rate={rate:.5f}")

plt.figure()
plt.plot(ns, rate_est, 'o-', label=r'empirical $-\frac{1}{n}\log P(\bar X_n\geq x^*)$')
plt.axhline(np.max(thetas*x_star - Lam), color='r', ls='--',
            label=fr'Cramér $I(x^*)={np.max(thetas*x_star - Lam):.4f}$')
plt.xlabel('n'); plt.ylabel('rate')
plt.title(fr'Large-deviation rate vs $n$  ($x^*={x_star}$, Uniform[-1,3])')
plt.legend(); plt.tight_layout(); plt.show()

print("\nAs n grows the empirical rate should approach I(x*).")


---
## Part 4 — Comparing the Three Bounds

| Tool | What it bounds | Tightness | Assumption |
|------|---------------|-----------|------------|
| **Hoeffding** | \(P(|\bar X_n-\mu|\geq t)\) | crude but universal | only boundedness \(a\leq X\leq b\) |
| **Chernoff (sub-Gaussian)** | \(P(\bar X_n\geq \mu+\varepsilon)\) | uses variance proxy \(\sigma^2\) | sub-Gaussian tails |
| **Cramér** | exact exponential rate \(I(x)\) | sharp asymptotic | i.i.d. + light-tailed MGF |

Hoeffding is the **coarsest** (it ignores distribution shape beyond the range); Chernoff uses the **MGF** and is tighter for sub-Gaussian variables; Cramér is the **limiting truth** — the exact exponential rate the other two upper-bound.

The code below plots all three for the same Uniform\([-1,3]\) setup at a fixed \(n\), so the gap between "universal bound" and "true rate" is visible.


In [ ]:
# Overlay Hoeffding, Chernoff (uniform-specific sub-Gaussian), and Cramer rate
a, b = -1.0, 3.0
mu = 0.5 * (a + b)
n = 300
ts = np.linspace(0.02, 1.4, 60)

# Hoeffding (two-sided)
hoeff = 2.0 * np.exp(-2.0 * n * ts**2 / (b - a)**2)

# Sub-Gaussian Chernoff (one-sided) with optimal sigma^2 = (b-a)^2/4 for Uniform
sigma2 = (b - a)**2 / 4.0
cher = np.exp(-n * ts**2 / (2 * sigma2))

# Cramer rate (one-sided): P(Xbar_n >= mu+t) ~ exp(-n * I(mu+t))
thetas = np.linspace(-4.0, 4.0, 2001)
Lam = np.empty_like(thetas)
small = np.abs(thetas) < 1e-8
Lam[small] = mu*thetas[small] + (b-a)**2*thetas[small]**2/24.0
t = thetas[~small]
Lam[~small] = np.log((np.exp(t*b) - np.exp(t*a)) / (t*(b-a)))
cramer_one = np.exp(-n * np.array([np.max(thetas*(mu+t) - Lam) for t in ts]))
cramer_two = 2 * cramer_one  # crude two-sided proxy for plotting

plt.figure()
plt.semilogy(ts, hoeff, 'k--', lw=2, label='Hoeffding (two-sided)')
plt.semilogy(ts, cher, 'b-.', lw=2, label=r'sub-Gaussian Chernoff $\exp(-nt^2/(2\sigma^2))$')
plt.semilogy(ts, np.minimum(cramer_two, 1.0), 'r-', lw=2, label=r'Cramér $\sim 2e^{-nI(\mu+t)}$')
plt.xlabel('t'); plt.ylabel(r'tail probability (log scale)')
plt.title(f'Bound comparison  (n={n}, Uniform[{a},{b}])')
plt.legend(); plt.tight_layout(); plt.show()

print("Hoeffding sits above Chernoff, which sits above the Cramer rate — the")
print("ordering reflects how much distributional information each bound exploits.")

# Your turn prompt
print("\nYour turn: replace the uniform with a truncated-exponential or bounded")
print("mixture, recompute Lambda numerically, and see how the Cramer rate changes.")


---
## Your turn

- **Hoeffding.** Halve the range \([a,b]\) and observe how much tighter the bound becomes for the same \(t\).
- **Chernoff.** Replace Bernoulli with a centered Rademacher (±1) variable. What is the optimal sub-Gaussian variance proxy?
- **Cramér.** Switch to \(X_i\sim\mathrm{Exp}(1)\) (light-tailed, unbounded). Compute \(\Lambda(\theta)=\log\tfrac{1}{1-\theta}\) for \(\theta<1\) and the rate function \(I(x)=x-1-\log x\). Verify the rate by simulation.
- **Compare.** For a fixed \(n\), plot the ratio Hoeffding/Cramér and Chernoff/Cramér as a function of \(t\); which bound is loosest near the mean vs. in the deep tail?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. **Hoeffding, two-sided.** Let \(X_i\sim\mathrm{Uniform}[-2,4]\) i.i.d. with \(n=250\). Bound \(P(|\bar X_n-\mu|\geq 0.5)\) using Hoeffding and compare to a simulation with 200,000 trials.

2. **Chernoff for Bernoulli.** For \(X_i\sim\mathrm{Bernoulli}(0.4)\) and \(n=500\), use the KL-divergence form of Chernoff to bound \(P(\bar X_n\geq 0.5)\). Compare to the sub-Gaussian bound with \(\sigma^2=1/4\) and to simulation.

3. **Cramér rate for uniform.** For \(X_i\sim\mathrm{Uniform}[0,1]\), compute the exact \(\Lambda(\theta)=\log\tfrac{e^\theta-1}{\theta}\) and the rate function \(I(x)\) on \(x\in(0,1)\). Plot \(I(x)\) and report \(I(0.8)\).

4. **Rare-event estimation.** Using importance sampling with a tilted distribution whose mean is the target \(x^*=0.8\) for Uniform\([0,1]\), estimate \((1/n)\log P(\bar X_n\geq 0.8)\) for \(n=50,100,200,400\) and compare to \(I(0.8)\).

5. **Bound comparison.** For \(n=300\) i.i.d. Uniform\([-1,3]\) variables, plot Hoeffding, sub-Gaussian Chernoff (with \(\sigma^2=(b-a)^2/4\)), and the Cramér rate as functions of \(t\), on a log scale. Comment on which bound is tightest near the mean versus in the deep tail.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(\\mu=1\\), range \\((b-a)^2=36\\). Hoeffding: \\(2\\exp(-2\\cdot250\\cdot0.25/36)=2e^{-3.472}\\approx0.0624\\). Simulation:

```python
import numpy as np
rng=np.random.default_rng(0)
x=rng.uniform(-2,4,size=(200000,250)).mean(1)
print(np.mean(np.abs(x-1)>=0.5))          # ~0.005 (much smaller — Hoeffding is loose)
```

**2.** KL form: \\(P(\\bar X_n\\geq0.5)\\leq e^{-n\\,D(0.5\\|0.4)}\\) with \\(D(0.5\\|0.4)=0.5\\ln(1.25)+0.5\\ln(0.75/0.4)\\approx0.02014\\), giving \\(e^{-10.07}\\approx4.2\\times10^{-5}\\). Sub-Gaussian: \\(e^{-500\\cdot0.1/2}=e^{-25}\\approx1.4\\times10^{-11}\\) — much tighter but only valid because Bernoulli is \\(1/4\\)-sub-Gaussian; here the KL-Chernoff is the tighter *correct* one for this threshold.

**3.** ```python
import numpy as np
th=np.linspace(-5,5,2001); th=th[np.abs(th)>1e-9]
Lam=np.log((np.exp(th)-1)/th)
xs=np.linspace(0.02,0.98,121)
I=np.array([np.max(th*x-Lam) for x in xs])
import matplotlib.pyplot as plt
plt.plot(xs,I); plt.axvline(0.5,color='r',ls=':'); plt.show()
print("I(0.8)=", np.max(th*0.8-Lam))   # ~0.0837
```

**4.** Tilt by exponential change of measure: sample from density \\(\\propto e^{\\theta x}f(x)\\) with \\(\\theta^\\*=\\arg\\max(\\theta\\cdot0.8-\\Lambda(\\theta))\\). Use likelihood ratios \\(e^{-\\theta^\\* S_n+n\\Lambda(\\theta^\\*)}\\) to estimate the tail. The estimate of \\((1/n)\\log P(\\bar X_n\\geq0.8)\\) approaches \\(-I(0.8)\\approx-0.0837\\).

```python
import numpy as np
from scipy.optimize import minimize_scalar
rng=np.random.default_rng(0)
a,b=0,1; x_star=0.8
# find theta* maximizing theta*x - Lambda(theta) for Uniform[0,1]
def neg(t): return -(t*x_star - np.log((np.exp(t)-1)/t if abs(t)>1e-9 else 0.5*t))
res=minimize_scalar(neg,bounds=(-5,5),method='bounded'); th=res.x
# sample X ~ tilted Uniform via inverse CDF of exp(theta x) on [0,1]
def sample(n,N):
    U=rng.uniform(size=(N,n))
    return np.log(1+U*(np.exp(th)-1))/th if abs(th)>1e-9 else U
for n in [50,100,200,400]:
    N=200000; X=sample(n,N); S=X.sum(1)
    w=np.exp(-th*S + n*(np.log((np.exp(th)-1)/th if abs(th)>1e-9 else 0.5*th)))
    phat=np.sum(w*(S/n>=x_star))/N
    print(n, -np.log(max(phat,1e-12))/n)
```

**5.** See the Part-4 code; the Cramér curve sits below both Hoeffding and Chernoff for all \\(t>0\\) (it is the *true* exponential rate). Hoeffding is loosest near the mean (it ignores variance structure); Chernoff is tighter but still off by a constant factor in the exponent; Cramér is asymptotically exact. In the deep tail the three converge in *shape* but Hoeffding's constant \\(2/(b-a)^2\\) in the exponent is pessimistic versus \\(I(x)\\).

</details>
